In [1]:
import os
os.chdir("/home/jp/Code/SenPa-MAE/src/") # adapt here to point to the src dir
os.listdir()

['__pycache__',
 'dataloader.py',
 'main.py',
 'responsefunctions',
 'load_model_checkpoint.ipynb',
 'configs',
 'metrics.py',
 'utils.py',
 'train.py',
 'model.py',
 'maskingfunction.py',
 '.ipynb_checkpoints']

In [3]:
import torch
import hydra
import omegaconf
import numpy as np
import rasterio as rio

In [4]:
# test signal with four channels:
with rio.open("./../data/example_s2_image.tif","r") as src:
    X = src.read([1,2,3,4])
    X = X/10000 # preprocessing
X = np.expand_dims(X,0) # add batch dim
X = X[:,:,0:144,0:144] # cut to right patch size
X = torch.tensor(X,dtype=torch.float32) # cast to tensor

# baseline vanilla MAE

In [5]:
# define locations
path_weights = "./../weights/basemae.pt"
path_config = "./configs/pretrain/basemae.yaml"

# grab the config and overwrite the masking step in the encoder
cfg = omegaconf.OmegaConf.load(path_config)
cfg.model.encoder.maskingfunction._target_ = "maskingfunction.DummyShuffle"

# init the encoder part of the model
model = hydra.utils.instantiate(cfg.model.encoder)

# load the corresponding weights and clean up the state dict as seen here:
# https://discuss.pytorch.org/t/how-to-save-load-a-model-with-torch-compile/179739
ckpt = torch.load(path_weights,map_location=torch.device('cpu'))
state_dict = {k.replace("_orig_mod.","").replace("encoder.",""): v for k, v in ckpt['model_state_dict'].items() if "encoder" in k}

# load the state dict
model.load_state_dict(state_dict)
model.eval()
pass

In [6]:
# no parameter encoding hence we can just process an image
features, _ = model(X)

In [7]:
features.shape # 324 features for 144x144 px image with 4 channels and 16 patch size (9*9*4=324)

torch.Size([324, 1, 768])

# SenPaMAE single parameter encoding step

In [8]:
# define locations
path_weights = "./../weights/senpamae_singleParameterEmbedding.pt"
path_config = "./configs/pretrain/senpamae_singleParameterEmbedding.yaml"

# grab the config and overwrite the masking step in the encoder
cfg = omegaconf.OmegaConf.load(path_config)
cfg.model.encoder.maskingfunction._target_ = "maskingfunction.DummyShuffle"

# init the encoder part of the model
model = hydra.utils.instantiate(cfg.model.encoder)

# load the corresponding weights and clean up the state dict as seen here:
# https://discuss.pytorch.org/t/how-to-save-load-a-model-with-torch-compile/179739
ckpt = torch.load(path_weights,map_location=torch.device('cpu'))
state_dict = {k.replace("_orig_mod.","").replace("encoder.",""): v for k, v in ckpt['model_state_dict'].items() if "encoder" in k}

# load the state dict
model.load_state_dict(state_dict)
model.eval()
pass

In [9]:
# we need to supply the responsefunctions and the gsd values of the signal

# for testing purposes we load the S2 responsefunctions 
responsefunctions = np.load(os.path.join("./responsefunctions/rfs_sentinel2_a.npy"))
responsefunctions = np.swapaxes(responsefunctions,0,1) # (10, 2301)
responsefunctions = torch.Tensor(responsefunctions[:4]).unsqueeze(0) # just consider the first four

# GSDs values for the four channel input
gsds = torch.Tensor([[10,10,20,5]])

In [10]:
features, _ = model(X, responsefunctions, gsds)

In [11]:
features.shape

torch.Size([324, 1, 768])

# SenPaMAE double parameter encoding step

In [12]:
# define locations
path_weights = "./../weights/senpamae_doubleParameterEmbedding.pt"
path_config = "./configs/pretrain/senpamae_doubleParameterEmbedding.yaml"

# grab the config and overwrite the masking step in the encoder
cfg = omegaconf.OmegaConf.load(path_config)
cfg.model.encoder.maskingfunction._target_ = "maskingfunction.DummyShuffle"

# init the encoder part of the model
model = hydra.utils.instantiate(cfg.model.encoder)

# load the corresponding weights and clean up the state dict as seen here:
# https://discuss.pytorch.org/t/how-to-save-load-a-model-with-torch-compile/179739
ckpt = torch.load(path_weights,map_location=torch.device('cpu'))
state_dict = {k.replace("_orig_mod.","").replace("encoder.",""): v for k, v in ckpt['model_state_dict'].items() if "encoder" in k}

# load the state dict
model.load_state_dict(state_dict)
model.eval()
pass

In [13]:
# we need to supply the responsefunctions and the gsd values of the signal

# for testing purposes we load the S2 responsefunctions 
responsefunctions = np.load(os.path.join("./responsefunctions/rfs_sentinel2_a.npy"))
responsefunctions = np.swapaxes(responsefunctions,0,1) # (10, 2301)
responsefunctions = torch.Tensor(responsefunctions[:4]).unsqueeze(0) # just consider the first four

# GSDs values for the four channel input
gsds = torch.Tensor([[10,10,20,5]])

In [14]:
features, _ = model(X, responsefunctions, gsds)

In [15]:
features.shape

torch.Size([324, 1, 768])